## Model Service

In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.calculation.calculation_service import CalculationService
from src.wind_turbines.wind_turbines_service import WindTurbinesService
from src.model.model_service import ModelService
from src.database.database_service import DatabaseService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os
import pandas as pd

/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [4]:
database_service = DatabaseService(cfg)

In [5]:
weather_station_service = WeatherStationService(cfg, database_service)
weather_stations_df = weather_station_service.load_from_database()

2025-09-08 21:52:38.225 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:223 - Loaded 26 weather stations from database


In [6]:
measurement_service = MeasurementService(cfg, database_service, weather_stations_df)

## Generate dataset

In [7]:
model_service = ModelService(cfg, database_service, measurement_service)

In [8]:
model_service.load_dataset()

2025-09-08 21:53:44.888 | ERROR    | src.model.model_service:load_dataset:48 - Error loading measurements from database: (psycopg.OperationalError) consuming input failed: SSL connection has been closed unexpectedly
[SQL: SELECT wind_station_measurements.id, wind_station_measurements.station_id, wind_station_measurements.quality_level, wind_station_measurements.average_wind_speed, wind_station_measurements.average_wind_direction, wind_station_measurements.air_pressure, wind_station_measurements.air_temperature_2m, wind_station_measurements.air_temperature_5cm, wind_station_measurements.relative_humidity, wind_station_measurements.dew_point_temperature, wind_station_measurements.precipitation_duration, wind_station_measurements.sum_precipitation_height, wind_station_measurements.precipitation_indicator, wind_station_measurements.record_date 
FROM wind_station_measurements]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
model_service.save_dataset_as_pickle()

NameError: name 'model_service' is not defined

In [4]:
df = pd.read_pickle("data/dataset.pkl")

In [6]:
df.columns

Index(['air_pressure', 'air_temperature_2m', 'air_temperature_5cm',
       'dew_point_temperature', 'id', 'precipitation_duration',
       'precipitation_indicator', 'quality_level', 'record_date',
       'relative_humidity', 'station_id', 'sum_precipitation_height', 'u', 'v',
       'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'u_lag_1', 'u_lag_3',
       'u_lag_6', 'u_lag_12', 'u_lag_24', 'v_lag_1', 'v_lag_3', 'v_lag_6',
       'v_lag_12', 'v_lag_24', 'u_roll_mean_3h', 'u_roll_std_3h',
       'u_roll_mean_6h', 'u_roll_std_6h', 'v_roll_mean_3h', 'v_roll_std_3h',
       'v_roll_mean_6h', 'v_roll_std_6h', 'pressure_tendency_3h',
       'pressure_tendency_6h', 'temperature_tendency_3h',
       'temperature_tendency_6h'],
      dtype='object')

In [5]:
from src.model.variant.auto_gluon_model import AutoGluonModel

model = AutoGluonModel()
model.train(df)


Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/Users/matthaei/Documents/code/python/bachelor-project/AutogluonModels/ag-20250909_071526'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jul 14 11:30:29 PDT 2025; root:xnu-11417.140.69~1/RELEASE_ARM64_T6000
CPU Count:          10
GPU Count:          1
Memory Avail:       11.12 GB / 32.00 GB (34.7%)
Disk Space Avail:   89.63 GB / 926.35 GB (9.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'freq': '10min',
 'hyperparameters': {'ARIMA': {},
                     'AutoARIMA': {},
                     'DirectTabular': {},
                     'ETS': {},
                     'Naive': {},
                     'RecursiveTabular': {},
                     'SeasonalNaive': {},
                     'Theta': {}},
 'known_cova

[50]	valid_set's l1: 0.0163225
[100]	valid_set's l1: 0.0123545
[150]	valid_set's l1: 0.0119434
[200]	valid_set's l1: 0.0118176
[250]	valid_set's l1: 0.0124802
[300]	valid_set's l1: 0.012696


Shortening all series to at most 125192
		-1.8847      = Validation score (-MAE)
		12.441  s    = Training runtime
		1.369   s    = Prediction runtime
	-1.8847       = Validation score (-MAE)
	12.72   s     = Training runtime
	1.37    s     = Validation (prediction) runtime
Reserving 96.6s for ensemble
Training timeseries model DirectTabular. Training for up to 96.6s of the 579.6s of remaining time.
	Window 0
	Time limit adjusted due to model hyperparameters: 96.42s -> 86.78s (ag.max_time_limit=None, ag.max_time_limit_ratio=0.9
Shortening all series to at most 125048
train_df shape: (999616, 55), val_df shape: (384, 55)


[50]	valid_set's l1: 0.608178
[100]	valid_set's l1: 0.535224
[150]	valid_set's l1: 0.528765
[200]	valid_set's l1: 0.532277
[250]	valid_set's l1: 0.53272


Shortening all series to at most 125096
Shortening all series to at most 125048
		-13.4383     = Validation score (-MAE)
		9.362   s    = Training runtime
		2.912   s    = Prediction runtime
	-13.4383      = Validation score (-MAE)
	9.64    s     = Training runtime
	2.91    s     = Validation (prediction) runtime
Reserving 113.4s for ensemble
Training timeseries model ETS. Training for up to 113.4s of the 567.1s of remaining time.
	Window 0
Shortening all time series to at most 2500
/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-package

[50]	valid_set's l1: 0.0686389
[100]	valid_set's l1: 0.0628757
[150]	valid_set's l1: 0.0589659
[200]	valid_set's l1: 0.0572093
[250]	valid_set's l1: 0.0557406
[300]	valid_set's l1: 0.0552294
[350]	valid_set's l1: 0.0545534
[400]	valid_set's l1: 0.054349
[450]	valid_set's l1: 0.0540352
[500]	valid_set's l1: 0.0537231
[550]	valid_set's l1: 0.0536171
[600]	valid_set's l1: 0.0533808
[650]	valid_set's l1: 0.0531809
[700]	valid_set's l1: 0.0530263
[750]	valid_set's l1: 0.0530079
[800]	valid_set's l1: 0.0527426
[850]	valid_set's l1: 0.052561
[900]	valid_set's l1: 0.0525048
[950]	valid_set's l1: 0.0525357
[1000]	valid_set's l1: 0.0524762
[1050]	valid_set's l1: 0.0524248
[1100]	valid_set's l1: 0.0524516
[1150]	valid_set's l1: 0.0524017
[1200]	valid_set's l1: 0.0523847
[1250]	valid_set's l1: 0.0522196
[1300]	valid_set's l1: 0.0521892
[1350]	valid_set's l1: 0.0522687
[1400]	valid_set's l1: 0.0521875
[1450]	valid_set's l1: 0.0521561
[1500]	valid_set's l1: 0.0522597
[1550]	valid_set's l1: 0.0522611

Shortening all series to at most 125192
		-0.9235      = Validation score (-MAE)
		38.533  s    = Training runtime
		1.318   s    = Prediction runtime
	-0.9235       = Validation score (-MAE)
	38.81   s     = Training runtime
	1.32    s     = Validation (prediction) runtime
Reserving 92.9s for ensemble
Training timeseries model DirectTabular. Training for up to 92.9s of the 557.7s of remaining time.
	Window 0
	Time limit adjusted due to model hyperparameters: 92.75s -> 83.48s (ag.max_time_limit=None, ag.max_time_limit_ratio=0.9
Shortening all series to at most 125048
train_df shape: (999616, 55), val_df shape: (384, 55)


[50]	valid_set's l1: 0.634448
[100]	valid_set's l1: 0.579544
[150]	valid_set's l1: 0.564537
[200]	valid_set's l1: 0.560657
[250]	valid_set's l1: 0.553567
[300]	valid_set's l1: 0.559761


Shortening all series to at most 125096
Shortening all series to at most 125048
		-2.4230      = Validation score (-MAE)
		11.013  s    = Training runtime
		2.846   s    = Prediction runtime
	-2.4230       = Validation score (-MAE)
	11.29   s     = Training runtime
	2.85    s     = Validation (prediction) runtime
Reserving 108.7s for ensemble
Training timeseries model ETS. Training for up to 108.7s of the 543.5s of remaining time.
	Window 0
Shortening all time series to at most 2500
		-0.7331      = Validation score (-MAE)
		0.160   s    = Training runtime
		2.881   s    = Prediction runtime
	-0.7331       = Validation score (-MAE)
	0.44    s     = Training runtime
	2.88    s     = Validation (prediction) runtime
Reserving 135.1s for ensemble
Training timeseries model Theta. Training for up to 135.1s of the 540.2s of remaining time.
	Window 0
Shortening all time series to at most 2500
		-1.0686      = Validation score (-MAE)
		0.166   s    = Training runtime
		0.408   s    = Prediction

In [6]:
model._predictors

{'u': <autogluon.timeseries.predictor.TimeSeriesPredictor at 0x16506d610>,
 'v': <autogluon.timeseries.predictor.TimeSeriesPredictor at 0x1653ed430>}